<a href="https://colab.research.google.com/github/oluwoleadetifa/movie_genre_classification/blob/dev/notebooks/05_bert_text_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git
%cd /content/movie_genre_classification

import sys
sys.path.append("/content/movie_genre_classification")

!pip install transformers sentencepiece torch scikit-learn joblib

Cloning into 'movie_genre_classification'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 166 (delta 78), reused 28 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 902.65 KiB | 4.14 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/movie_genre_classification


In [3]:
!pip install transformers sentencepiece torch scikit-learn joblib

In [7]:
import joblib
from sklearn.preprocessing import LabelEncoder

from src.config import TEXT_COLUMN, LABEL_COLUMN, MODELS_DIR
from src.data.load_data import load_dataset
from src.data.make_splits import create_splits, save_split_ids
from src.data.preprocess_text import preprocess_text_dataframe

# Load and preprocess dataset
df = load_dataset()
df = preprocess_text_dataframe(
    df,
    text_column=TEXT_COLUMN,
    output_column="clean_text"
)

# Create splits
train_df, val_df, test_df = create_splits(df)
save_split_ids(train_df, val_df, test_df)

# Create label encoder
label_encoder = LabelEncoder()
label_encoder.fit(train_df[LABEL_COLUMN])

joblib.dump(label_encoder, MODELS_DIR / "label_encoder.joblib")

print("Splits and label encoder created successfully.")

Splits and label encoder created successfully.


In [8]:
train_df, val_df, test_df = load_saved_splits()

train_df = preprocess_text_dataframe(
    train_df,
    text_column=TEXT_COLUMN,
    output_column="clean_text"
)

val_df = preprocess_text_dataframe(
    val_df,
    text_column=TEXT_COLUMN,
    output_column="clean_text"
)

test_df = preprocess_text_dataframe(
    test_df,
    text_column=TEXT_COLUMN,
    output_column="clean_text"
)

label_encoder = joblib.load(MODELS_DIR / "label_encoder.joblib")

y_train = label_encoder.transform(train_df[LABEL_COLUMN])
y_val = label_encoder.transform(val_df[LABEL_COLUMN])
y_test = label_encoder.transform(test_df[LABEL_COLUMN])

print(train_df.shape, val_df.shape, test_df.shape)

(600, 4) (200, 4) (200, 4)


In [9]:
embedder = BertEmbedder(
    model_name="distilbert-base-uncased",
    max_length=256
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
sample_embeddings = embedder.encode_texts(
    train_df["clean_text"].tolist()[:8],
    batch_size=4
)

print(sample_embeddings.shape)

100%|██████████| 2/2 [00:02<00:00,  1.46s/it]

(8, 768)


In [11]:
X_train_bert = embedder.encode_texts(
    train_df["clean_text"].tolist(),
    batch_size=16
)

X_val_bert = embedder.encode_texts(
    val_df["clean_text"].tolist(),
    batch_size=16
)

X_test_bert = embedder.encode_texts(
    test_df["clean_text"].tolist(),
    batch_size=16
)

print(X_train_bert.shape, X_val_bert.shape, X_test_bert.shape)

100%|██████████| 13/13 [01:16<00:00,  5.85s/it]

(600, 768) (200, 768) (200, 768)


In [12]:
bert_model = build_bert_classifier()
bert_model = train_text_model(bert_model, X_train_bert, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


In [13]:
y_val_pred_bert, y_val_prob_bert = predict_text_model(
    bert_model,
    X_val_bert
)

val_metrics_bert = evaluate_multiclass(
    y_val,
    y_val_pred_bert
)

print(val_metrics_bert)

{'accuracy': 0.76, 'macro_f1': 0.7555382100525941, 'macro_precision': 0.7633928571428571, 'macro_recall': 0.76, 'confusion_matrix': [[34, 3, 12, 1], [8, 32, 3, 7], [0, 0, 48, 2], [2, 7, 3, 38]]}


In [14]:
y_test_pred_bert, y_test_prob_bert = predict_text_model(
    bert_model,
    X_test_bert
)

test_metrics_bert = evaluate_multiclass(
    y_test,
    y_test_pred_bert
)

print(test_metrics_bert)

{'accuracy': 0.78, 'macro_f1': 0.7809008451865594, 'macro_precision': 0.783033781694496, 'macro_recall': 0.78, 'confusion_matrix': [[43, 3, 3, 1], [1, 34, 3, 12], [3, 4, 40, 3], [1, 8, 2, 39]]}


In [15]:
joblib.dump(
    bert_model,
    MODELS_DIR / "bert_text_model.joblib"
)

np.save(MODELS_DIR / "X_train_bert.npy", X_train_bert)
np.save(MODELS_DIR / "X_val_bert.npy", X_val_bert)
np.save(MODELS_DIR / "X_test_bert.npy", X_test_bert)

np.save(MODELS_DIR / "y_val_text_probs_bert.npy", y_val_prob_bert)
np.save(MODELS_DIR / "y_test_text_probs_bert.npy", y_test_prob_bert)

results_bert = {
    "val_metrics": val_metrics_bert,
    "test_metrics": test_metrics_bert,
    "classes": label_encoder.classes_.tolist()
}

with open(METRICS_DIR / "bert_results.json", "w") as f:
    json.dump(results_bert, f, indent=2)

print("BERT results saved.")

BERT results saved.
